# Gyara: fine-tune Hausa-ASR on your corrected data (free T4)

1. Upload a dataset exported from the Gyara correction app (`gyara export`, zipped).
2. Training refuses to start if any clip or **speaker** also appears in a
   held-out manifest (FLEURS test, plus your own held-out set if you have one).
   Do not weaken that check to make a run go.
3. Small-data defaults: LR 1e-5, 10% warmup, encoder frozen under 5 h,
   light SpecAugment, early stopping on a speaker-disjoint dev split
   (checkpoints are picked on dev WER, never on test).
4. Base and tuned models are evaluated **once** on held-out data and compared
   with a paired bootstrap. The result is printed as found, gain or not.

Bump `attempt` in the config every time you change a setting and re-train:
the count goes in the model card and `before_after.md` (we report how many
configs were tried).

The last cell zips the results (never audio or weights) and downloads them.

*N-ATLAS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.*

## 0. Setup

In [ ]:
# ---- The only settings you may need to change ---------------------------------
GYARA_OWNER = "Baffaabba"          # GitHub user that hosts the repo
GYARA_BRANCH = "feat/foundation"   # change to "main" once merged
GYARA_EXTRAS = "train"
# --------------------------------------------------------------------------------
GYARA_GIT = f"git+https://github.com/{GYARA_OWNER}/gyara@{GYARA_BRANCH}"

import importlib, importlib.util, os, pathlib, subprocess, sys

# Everything (data/, runs/) is written under WORK, so paths match the repo layout.
for base in ("/content", "/kaggle/working"):
    if os.path.isdir(base):
        WORK = pathlib.Path(base) / "gyara-work"
        break
else:
    WORK = pathlib.Path.cwd() / "gyara-work"
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def pip_install(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-3000:])
    return r.returncode == 0

INSTALLED_FROM = None
if importlib.util.find_spec("gyara"):
    INSTALLED_FROM = "already installed in this session"
elif pip_install(f"gyara[{GYARA_EXTRAS}] @ {GYARA_GIT}"):
    INSTALLED_FROM = GYARA_GIT
else:
    print("\n>>> GitHub install FAILED (repo not pushed yet, wrong branch, or private). "
          "Run the next cell to install from a zip instead.")
print("working dir:", WORK)
print("gyara installed from:", INSTALLED_FROM)

In [ ]:
# Fallback, ONLY if the cell above failed: install from a zip of the repo.
# On the laptop, in PowerShell, from E:\Repos\gyara:
#   Compress-Archive -Path gyara, pyproject.toml, README.md -DestinationPath gyara-src.zip -Force
# Then run this cell and pick gyara-src.zip. (Kaggle: add the zip as a dataset and set ZIP_PATH.)
ZIP_PATH = None
if INSTALLED_FROM is None:
    import zipfile
    if ZIP_PATH is None:
        from google.colab import files
        ZIP_PATH = next(iter(files.upload()))
    SRC = pathlib.Path("/content/gyara-src" if os.path.isdir("/content") else WORK.parent / "gyara-src")
    zipfile.ZipFile(ZIP_PATH).extractall(SRC)
    roots = sorted(p.parent for p in SRC.rglob("pyproject.toml"))
    assert roots, "no pyproject.toml in the zip: zip the repo root (gyara/, pyproject.toml, README.md)"
    assert pip_install("-e", f"{roots[0]}[{GYARA_EXTRAS}]"), "zip install failed; see pip output"
    sys.path.insert(0, str(roots[0]))  # editable installs are only importable after a restart
    importlib.invalidate_caches()
    INSTALLED_FROM = f"zip upload: {ZIP_PATH}"
print("gyara installed from:", INSTALLED_FROM)
!gyara --version

In [ ]:
# NCAIR1/* models are gated: accept the terms on each model page on huggingface.co
# (NCAIR1/Hausa-ASR and NCAIR1/N-ATLaS) with the account that owns the token, then
# store a read token as a secret named HF_TOKEN
# (Colab: key icon in the left bar, and switch on "Notebook access"; Kaggle: Add-ons > Secrets).
import json, os, platform, zipfile, glob
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
assert token, "No HF_TOKEN secret found"
os.environ["HF_TOKEN"] = token
from huggingface_hub import login
login(token=token)

import torch, transformers
import gyara
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
os.makedirs("runs/finetune/env", exist_ok=True)
try:
    from importlib.metadata import distribution
    direct_url = json.loads(distribution("gyara").read_text("direct_url.json") or "{}")
except Exception:
    direct_url = {}
ENV = {
    "gyara_version": gyara.__version__,
    "installed_from": INSTALLED_FROM,
    "git_commit": direct_url.get("vcs_info", {}).get("commit_id"),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "python": platform.python_version(),
}
json.dump(ENV, open("runs/finetune/env/env.json", "w"), indent=2)
print(json.dumps(ENV, indent=2))
!nvidia-smi | tee runs/finetune/env/nvidia-smi.txt

# Small text results only: never audio, never model weights.
KEEP = {".json", ".jsonl", ".md", ".srt", ".vtt", ".txt", ".yaml"}
def save_results(zip_name="gyara-results.zip", download=True):
    """Zip every small result file under runs/ (paths kept, so unzip at the repo root)."""
    files = [p for p in pathlib.Path("runs").rglob("*")
             if p.is_file() and p.suffix in KEEP
             and not any(part.startswith("checkpoint-") for part in p.parts)
             and ("model" not in p.parts or p.name == "README.md")]  # model card only
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, p.as_posix())
    print(f"{zip_name}: {len(files)} files")
    for p in sorted(files):
        print("  ", p.as_posix())
    if download:
        try:
            from google.colab import files as colab_files
            colab_files.download(zip_name)
        except Exception:
            print(f"Not on Colab: download {WORK / zip_name} from the file browser / Output tab.")

## 1. Held-out and mixing data (FLEURS ha_ng)

In [ ]:
!gyara fetch-fleurs --split test
!gyara fetch-fleurs --split train
TEST, FLEURS_TRAIN = "data/fleurs/test.jsonl", "data/fleurs/train.jsonl"
assert os.path.exists(TEST) and os.path.exists(FLEURS_TRAIN)

## 2. Upload your Gyara export (zip)

In [ ]:
EXPORT_ZIP = None  # Kaggle: set to the path of the zip under /kaggle/input/...
if EXPORT_ZIP is None:
    from google.colab import files
    EXPORT_ZIP = next(iter(files.upload()))
EXPORT_DIR = pathlib.Path("exports") / pathlib.Path(EXPORT_ZIP).stem
zipfile.ZipFile(EXPORT_ZIP).extractall(EXPORT_DIR)
# The export may be zipped with or without its top-level folder. The trainer reads
# manifest.jsonl, or the HF metadata.jsonl if that is all there is.
hits = list(EXPORT_DIR.rglob("manifest.jsonl")) or list(EXPORT_DIR.rglob("metadata.jsonl"))
assert hits, "no manifest.jsonl or metadata.jsonl in the zip: is this a gyara export?"
EXPORT_DIR = hits[0].parent
print(EXPORT_DIR, sum(1 for _ in open(hits[0], encoding="utf-8")), "rows in", hits[0].name)

## 3. Config

`LANGUAGE` must match the benchmark run: use `chosen_language` from
`runs/g0-smoke/decoding_choice.json` (written by `benchmark_colab.ipynb`).
`"auto"` there means `None` here.

In [ ]:
import yaml
LANGUAGE = "hausa"   # or None if decoding_choice.json chose "auto"
OWN_HELDOUT = None   # optional: path to your own frozen held-out manifest (>= 30 min, other speakers)
config = {
    "attempt": 1,                     # bump on every re-train with changed settings
    "base_model": "NCAIR1/Hausa-ASR",
    "language": LANGUAGE,
    "output_dir": "runs/finetune/hausa-asr-gyara",
    "model_name": "hausa-asr-gyara",
    "data": {
        "exports": [str(EXPORT_DIR)],
        "fleurs_fraction": 0.3,       # 30% FLEURS train to limit forgetting
        "fleurs_manifest": FLEURS_TRAIN,
        "heldout": [TEST] + ([OWN_HELDOUT] if OWN_HELDOUT else []),
        "dev_fraction": 0.1,
    },
    "training": {"max_steps": 1000, "eval_steps": 100, "per_device_train_batch_size": 8,
                 "gradient_accumulation_steps": 2},
    "eval": {"after": True},
}
os.makedirs("runs/finetune/hausa-asr-gyara", exist_ok=True)
yaml.safe_dump(config, open("finetune.yaml", "w"), sort_keys=False, allow_unicode=True)
yaml.safe_dump(config, open("runs/finetune/hausa-asr-gyara/config.yaml", "w"), sort_keys=False, allow_unicode=True)
print(open("finetune.yaml").read())

## 4. Train (leakage check first)

In [ ]:
!gyara finetune --config finetune.yaml
save_results("gyara-finetune-results.zip", download=False)  # survives a disconnect of the next cells

## 5. Before / after on held-out data

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("runs/finetune/hausa-asr-gyara/before_after.md", encoding="utf-8").read()))

In [ ]:
# The same paired comparison through the CLI, for the FLEURS test set.
# Writes compare.md + compare.json into the tuned run's folder.
tuned = sorted(glob.glob("runs/finetune/hausa-asr-gyara/eval/hausa-asr-gyara-a*-test"))
base = sorted(glob.glob("runs/finetune/hausa-asr-gyara/eval/base-test"))
print(tuned, base)
if tuned and base:
    !gyara compare {tuned[-1]} {base[0]} > /dev/null
    display(Markdown(open(f"{tuned[-1]}/compare.md", encoding="utf-8").read()))

## 6. (Optional) publish

The model keeps the N-ATLAS licence of its base (free use capped at 1,000
active users; a renamed derivative must say "Powered by Awarri"). The model
card in `runs/finetune/hausa-asr-gyara/model/README.md` already says so.

In [ ]:
# from huggingface_hub import HfApi
# HfApi().upload_folder(folder_path="runs/finetune/hausa-asr-gyara/model",
#                       repo_id="<your-user>/hausa-asr-gyara-powered-by-awarri", private=True)
print(open("runs/finetune/hausa-asr-gyara/model/README.md", encoding="utf-8").read())

## 7. Save the results

Zips `runs/` text results (no audio, no weights, no checkpoints) and downloads
them. Unzip at the repo root: you get `runs/finetune/hausa-asr-gyara/`
(`before_after.md`, `summary.json`, `config.yaml`, `model/README.md`, and
`eval/*/metrics.json`, `report.md`, `meta.json`, `compare.json`, `compare.md`,
`predictions.jsonl`) plus `runs/finetune/env/` (GPU, versions, install source). The weights stay in
`runs/finetune/hausa-asr-gyara/model/`: publish them (step 6) or copy them to
Drive before the session ends.

In [ ]:
save_results("gyara-finetune-results.zip")